In [ ]:
from pathlib import Path

LEGALIR_SOURCE_PATH = Path("/kaggle/input/datasets/mduy2911/legalir/train.json")
CORPUS_PATH = Path("/kaggle/input/datasets/mduy2911/legalir/selected-contexts")
DENSE_MODEL_PATH = Path("/kaggle/input/datasets/mduy2911/bge-m3-kaggle")
DENSE_MODEL_NAME = "BAAI/bge-m3"
DENSE_DECLARED_REVISION = "5617a9f61b028005a4858fdac845db406aefb181"
EXPECTED_SOURCE_SHA256 = "c39cde9e74977e350f1456e7d487aafe67d2bcbaa4fa26fcabd557fe635635b7"
EXPECTED_SPLIT_COUNTS = {"train": 4_941, "dev": 1_036, "holdout": 1_023}
EXPECTED_DOCUMENTS = 8_532
EXPECTED_FIXED_CHUNKS = 199_816
CHUNK_SIZE = 2_000
OVERLAP_VARIANTS = {"overlap_0": 0, "overlap_200_control": 200, "overlap_500": 500}
TOP_K_CHUNKS = 2_000
DENSE_MAX_LENGTH = 8_192
CORPUS_BATCH_SIZE = 256
QUERY_BATCH_SIZE = 64
FINAL_K = 5
EVALUATION_DEPTHS = (10, 20, 50, 100, 200)
BASELINE_TOLERANCE = 1e-3
TARGET_GPU = "RTX PRO 6000 / up to 96GB VRAM"
DENSE_BASELINE = {
    "recall_at_10": 0.9227799227799228, "recall_at_20": 0.9497265122265123,
    "recall_at_50": 0.9769144144144144, "recall_at_100": 0.9819015444015444,
    "recall_at_200": 0.9877734877734878, "mrr": 0.7159583402955311,
}
RESEARCH_AXIS = "overlap amount at fixed chunk_size=2000"
CONTROL = "2000/overlap200"
INDEPENDENT_VARIABLE = {"overlap_characters": [0, 200, 500]}
FIXED_CONTROLS = {
    "chunk_size": 2000, "dense_model": DENSE_MODEL_NAME, "top_k_chunks": 2000,
    "document_aggregation": "sum top2", "reranking": None,
}
DECISION_RULE = "Retrieval-only DEV diagnostic; no automatic holdout decision. Determine only whether overlap merits later downstream evaluation."
RESULT_PATH = Path("/kaggle/working/fixed_window_overlap_dense_dev_results.json")


In [ ]:
# Enforce Internet-OFF execution and fail loudly for missing attached artifacts.
import os

os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

required = [
    (LEGALIR_SOURCE_PATH, "LegalIR source JSON", False),
    (CORPUS_PATH, "LegalIR corpus directory", True),
    (DENSE_MODEL_PATH, "complete local BGE-M3 snapshot", True),
]
for path, description, must_be_directory in required:
    exists = path.is_dir() if must_be_directory else path.is_file()
    if not exists:
        raise FileNotFoundError(f"Attach the {description} at: {path}")


In [ ]:
# Standalone fixed-DEV LegalIR implementation. No repository runtime is imported.
import gc
import json
from collections import Counter, defaultdict
from hashlib import sha256
from math import isfinite
from statistics import median
from time import perf_counter

import numpy as np
import torch
import torch.nn.functional as F
import transformers
from transformers import AutoModel, AutoTokenizer


def read_json(path: Path):
    try:
        with path.open(encoding="utf-8-sig") as stream:
            return json.load(stream)
    except json.JSONDecodeError as exc:
        raise ValueError(f"{path}: invalid JSON: {exc}") from exc


def load_fixed_dev(path: Path) -> tuple[dict, dict]:
    source_digest = sha256(path.read_bytes()).hexdigest()
    if source_digest != EXPECTED_SOURCE_SHA256:
        raise RuntimeError(
            f"LegalIR source SHA-256 mismatch: expected {EXPECTED_SOURCE_SHA256}, "
            f"got {source_digest}. Stop before reading samples."
        )
    value = read_json(path)
    if not isinstance(value, dict) or not all(isinstance(v, dict) for v in value.values()):
        raise ValueError(f"{path}: expected an object keyed by sample ID")
    samples = {str(sample_id): sample for sample_id, sample in value.items()}
    if len(samples) != len(value):
        raise ValueError("duplicate sample IDs after string canonicalization")

    split_counts = {"train": 0, "dev": 0, "holdout": 0}
    dev_ids = []
    for sample_id, sample in samples.items():
        question = sample.get("question")
        group_key = question if isinstance(question, str) else f"\0fallback-sample-id:{sample_id}"
        bucket = int(sha256(group_key.encode("utf-8")).hexdigest()[:8], 16) % 100
        partition = "train" if bucket < 70 else "dev" if bucket < 85 else "holdout"
        split_counts[partition] += 1
        if partition == "dev":
            dev_ids.append(sample_id)
    if split_counts != EXPECTED_SPLIT_COUNTS:
        raise RuntimeError(f"fixed split counts mismatch: {split_counts}; stop")

    dev_ids.sort()
    dev = {sample_id: samples[sample_id] for sample_id in dev_ids}
    del samples, value
    for sample_id, sample in dev.items():
        if not isinstance(sample.get("question"), str):
            raise TypeError(f"DEV sample {sample_id!r}: question must be a string")
        answer = sample.get("answer")
        if not isinstance(answer, list) or not answer:
            raise ValueError(f"DEV sample {sample_id!r}: expected a non-empty answer list")
        canonical = [str(document_id) for document_id in answer]
        if len(canonical) != len(set(canonical)):
            raise ValueError(f"DEV sample {sample_id!r}: answer contains duplicate IDs")
    evaluated_partition = "dev"
    if evaluated_partition != "dev" or len(dev) != EXPECTED_SPLIT_COUNTS["dev"]:
        raise RuntimeError("this notebook may evaluate fixed DEV only")
    return dev, {
        "version": "legalir_split_v1",
        "evaluated_partition": evaluated_partition,
        "queries": len(dev),
        "source_sha256": source_digest,
        "split_counts": split_counts,
        "selection_only": True,
        "holdout_metrics_computed": False,
    }


def load_corpus(path: Path) -> list[dict]:
    paths = sorted(item for item in path.rglob("*") if item.is_file() and item.suffix.lower() == ".json")
    if not paths:
        raise ValueError(f"{path}: corpus directory contains no JSON files")
    documents = []
    for json_path in paths:
        value = read_json(json_path)
        values = value if isinstance(value, list) else [value]
        if not all(isinstance(document, dict) for document in values):
            raise ValueError(f"{json_path}: expected document object(s)")
        documents.extend(values)
    document_ids = []
    for document in documents:
        if document.get("id") is None:
            raise ValueError("corpus document is missing a non-null ID")
        document_id = str(document["id"])
        if not isinstance(document.get("passage"), str):
            raise TypeError(f"document {document_id!r}: passage must be a string")
        document_ids.append(document_id)
    if len(document_ids) != len(set(document_ids)):
        raise ValueError("corpus contains duplicate document IDs")
    if len(documents) != EXPECTED_DOCUMENTS:
        raise RuntimeError(f"expected {EXPECTED_DOCUMENTS} documents, got {len(documents)}")
    return documents


def distribution(values: list[int]) -> dict:
    array = np.asarray(values, dtype=np.int64)
    if array.size == 0:
        return {"min": None, "median": None, "p95": None, "max": None}
    return {
        "min": int(array.min()),
        "median": float(np.median(array)),
        "p95": float(np.percentile(array, 95)),
        "max": int(array.max()),
    }


def validate_chunk_provenance(documents: list[dict], chunks: list[dict]) -> dict:
    source_by_id = {str(document["id"]): document["passage"] for document in documents}
    chunk_ids = [chunk["chunk_id"] for chunk in chunks]
    if len(chunk_ids) != len(set(chunk_ids)):
        raise RuntimeError("chunk IDs must be unique")
    intervals = defaultdict(list)
    for chunk in chunks:
        document_id = chunk["document_id"]
        source = source_by_id.get(document_id)
        if source is None:
            raise RuntimeError("chunk references an unknown document")
        start, end = chunk["char_start"], chunk["char_end"]
        if not (0 <= start < end <= len(source)) or chunk["text"] != source[start:end]:
            raise RuntimeError(f"chunk {chunk['chunk_id']!r}: exact provenance failed")
        intervals[document_id].append((start, end))
    for document_id, source in source_by_id.items():
        if not source:
            continue
        ordered = sorted(intervals[document_id])
        if not ordered or ordered[0][0] != 0:
            raise RuntimeError(f"document {document_id!r}: coverage does not start at zero")
        covered_end = 0
        for start, end in ordered:
            if start > covered_end:
                raise RuntimeError(f"document {document_id!r}: chunk coverage gap")
            covered_end = max(covered_end, end)
        if covered_end != len(source):
            raise RuntimeError(f"document {document_id!r}: incomplete source coverage")
    return {
        "exact_source_slices": True,
        "unique_chunk_ids": True,
        "complete_non_empty_source_coverage": True,
    }


def fixed_window_chunks(documents: list[dict], chunk_size: int, overlap: int) -> tuple[list[dict], dict]:
    if chunk_size <= 0 or overlap < 0 or overlap >= chunk_size:
        raise ValueError("invalid fixed-window parameters")
    step = chunk_size - overlap
    chunks = []
    counts = []
    for document in documents:
        document_id = str(document["id"])
        source = document["passage"]
        before = len(chunks)
        for chunk_index, start in enumerate(range(0, len(source), step)):
            end = min(start + chunk_size, len(source))
            chunks.append({
                "chunk_id": f"{document_id}:{chunk_index}",
                "document_id": document_id,
                "chunk_index": chunk_index,
                "text": source[start:end],
                "char_start": start,
                "char_end": end,
            })
            if end == len(source):
                break
        counts.append(len(chunks) - before)
    provenance = validate_chunk_provenance(documents, chunks)
    return chunks, {
        "number_of_chunks": len(chunks),
        "chunk_size": chunk_size,
        "overlap": overlap,
        "step": step,
        "chunk_length_characters": distribution([len(chunk["text"]) for chunk in chunks]),
        "chunks_per_document": distribution(counts),
        "provenance": provenance,
    }


def model_metadata(model, model_name: str, declared_revision: str, path: Path) -> dict:
    value = getattr(model.config, "_commit_hash", None)
    config_hash = value.strip() if isinstance(value, str) and value.strip() else None
    if config_hash is not None and config_hash != declared_revision:
        raise RuntimeError(
            f"{model_name} config revision {config_hash!r} != declared {declared_revision!r}"
        )
    return {
        "model_name": model_name,
        "declared_revision": declared_revision,
        "config_commit_hash": config_hash,
        "revision_status": "verified-from-config" if config_hash else "declared-offline-snapshot",
        "local_input_path": str(path),
    }


def load_dense_model() -> dict:
    if not torch.cuda.is_available():
        raise RuntimeError("Enable a Kaggle CUDA accelerator")
    started = perf_counter()
    tokenizer = AutoTokenizer.from_pretrained(DENSE_MODEL_PATH, local_files_only=True)
    model = AutoModel.from_pretrained(DENSE_MODEL_PATH, dtype=torch.float16, local_files_only=True)
    if int(getattr(model.config, "max_position_embeddings", 0)) < DENSE_MAX_LENGTH:
        raise RuntimeError("local dense model does not support max_length=8192")
    metadata = model_metadata(model, DENSE_MODEL_NAME, DENSE_DECLARED_REVISION, DENSE_MODEL_PATH)
    model.to("cuda").eval()
    return {"tokenizer": tokenizer, "model": model, "metadata": metadata, "load_seconds": perf_counter() - started}


def encode_normalized_cls(model_bundle: dict, texts: list[str], batch_size: int) -> dict:
    embeddings = []
    started = perf_counter()
    for start in range(0, len(texts), batch_size):
        batch = texts[start:start + batch_size]
        inputs = model_bundle["tokenizer"](
            batch, padding=True, truncation=True, max_length=DENSE_MAX_LENGTH, return_tensors="pt"
        )
        inputs = {name: value.to("cuda") for name, value in inputs.items()}
        with torch.no_grad():
            output = model_bundle["model"](**inputs, return_dict=True)
            embedding = F.normalize(output.last_hidden_state[:, 0], p=2, dim=1)
        if embedding.ndim != 2 or not torch.isfinite(embedding).all():
            raise RuntimeError("dense encoder returned invalid CLS embeddings")
        embeddings.append(embedding.cpu())
    encoded = torch.cat(embeddings, dim=0)
    if encoded.shape[0] != len(texts):
        raise RuntimeError("dense embedding count mismatch")
    return {"embeddings": encoded, "seconds": perf_counter() - started}


def retrieve_dense_hits(query_embeddings: torch.Tensor, corpus_embeddings: torch.Tensor, chunks: list[dict], sample_ids: list[str]) -> dict:
    if query_embeddings.shape[0] != len(sample_ids) or corpus_embeddings.shape[0] != len(chunks):
        raise ValueError("embedding count mismatch")
    if len(chunks) < TOP_K_CHUNKS:
        raise ValueError("corpus has fewer chunks than requested retrieval depth")
    started = perf_counter()
    corpus_gpu = corpus_embeddings.to("cuda")
    hits = {}
    for start in range(0, len(sample_ids), QUERY_BATCH_SIZE):
        batch_ids = sample_ids[start:start + QUERY_BATCH_SIZE]
        query_gpu = query_embeddings[start:start + len(batch_ids)].to("cuda")
        similarities = query_gpu @ corpus_gpu.T
        if not torch.isfinite(similarities).all():
            raise RuntimeError("dense similarity contains non-finite values")
        scores, indices = torch.topk(similarities, k=TOP_K_CHUNKS, dim=1, sorted=True)
        for row, sample_id in enumerate(batch_ids):
            ordered = list(zip(scores[row].float().cpu().tolist(), indices[row].cpu().tolist()))
            ordered.sort(key=lambda item: (-item[0], item[1]))
            hits[sample_id] = [
                {
                    "chunk_index": int(chunk_index),
                    "document_id": chunks[int(chunk_index)]["document_id"],
                    "score": float(score),
                    "chunk_rank": rank,
                }
                for rank, (score, chunk_index) in enumerate(ordered, start=1)
            ]
    torch.cuda.synchronize()
    seconds = perf_counter() - started
    del corpus_gpu
    torch.cuda.empty_cache()
    return {"hits": hits, "seconds": seconds}


def dense_aggregate(scores: list[float], rule: str) -> float:
    ordered = sorted(scores, reverse=True)
    if rule == "max_top1":
        return ordered[0]
    if rule == "sum_top2":
        return sum(ordered[:2])
    if rule == "mean_top2":
        selected = ordered[:2]
        return sum(selected) / len(selected)
    if rule == "sum_top3":
        return sum(ordered[:3])
    raise ValueError(f"unknown dense aggregation rule: {rule}")


def candidates_from_hits(
    hits_by_query: dict,
    rule: str,
    depth: int,
    support_limit: int = 8,
    require_exact_depth: bool = True,
) -> dict:
    output = {}
    for sample_id, hits in hits_by_query.items():
        grouped = defaultdict(list)
        for hit in hits:
            if not isfinite(hit["score"]):
                raise RuntimeError("non-finite dense hit")
            grouped[hit["document_id"]].append(hit)
        documents = []
        for document_id, document_hits in grouped.items():
            ordered = sorted(document_hits, key=lambda item: (-item["score"], item["chunk_rank"], item["chunk_index"]))
            documents.append({
                "document_id": document_id,
                "dense_document_score": dense_aggregate([item["score"] for item in ordered], rule),
                "best_chunk_rank": ordered[0]["chunk_rank"],
                "available_global_hits": len(ordered),
                "supporting_chunk_indices": [item["chunk_index"] for item in ordered[:support_limit]],
            })
        documents.sort(key=lambda item: (-item["dense_document_score"], item["best_chunk_rank"], item["document_id"]))
        selected = documents[:depth]
        if not selected:
            raise RuntimeError(f"sample {sample_id!r}: dense retrieval produced no candidates")
        if require_exact_depth and len(selected) != depth:
            raise RuntimeError(f"sample {sample_id!r}: expected {depth} candidates")
        for rank, document in enumerate(selected, start=1):
            document["original_dense_rank"] = rank
        ids = [document["document_id"] for document in selected]
        if len(ids) != len(set(ids)):
            raise RuntimeError("candidate ranking contains duplicate document IDs")
        output[sample_id] = selected
    return output


def rankings_from_candidates(candidates: dict) -> dict:
    return {
        sample_id: [document["document_id"] for document in documents]
        for sample_id, documents in candidates.items()
    }


def evaluate_rankings(samples: dict, rankings: dict, depths=(5, 10, 20, 50, 100)) -> dict:
    if set(samples) != set(rankings):
        raise RuntimeError("ranking IDs do not match fixed DEV IDs")
    recalls = {depth: [] for depth in depths}
    reciprocal_ranks = []
    precision_5, recall_5 = [], []
    for sample_id, sample in samples.items():
        ranked = [str(document_id) for document_id in rankings[sample_id]]
        if len(ranked) != len(set(ranked)):
            raise RuntimeError(f"sample {sample_id!r}: duplicate ranked document IDs")
        gold = {str(document_id) for document_id in sample["answer"]}
        for depth in depths:
            effective_k = min(depth, len(ranked))
            recalls[depth].append(len(gold.intersection(ranked[:effective_k])) / len(gold))
        first = next((rank for rank, document_id in enumerate(ranked, 1) if document_id in gold), None)
        reciprocal_ranks.append(0.0 if first is None else 1.0 / first)
        predicted = ranked[:FINAL_K]
        if not 1 <= len(predicted) <= 5:
            raise RuntimeError("prediction length must be between one and five")
        overlap = len(gold.intersection(predicted))
        precision_5.append(overlap / len(predicted))
        recall_5.append(overlap / len(gold))
    return {
        "precision": float(np.mean(precision_5)),
        "recall": float(np.mean(recall_5)),
        "mrr": float(np.mean(reciprocal_ranks)),
        **{f"recall_at_{depth}": float(np.mean(values)) for depth, values in recalls.items()},
    }


def first_gold_bins(samples: dict, rankings: dict) -> dict:
    bins = Counter()
    found = []
    for sample_id, sample in samples.items():
        gold = {str(document_id) for document_id in sample["answer"]}
        first = next((rank for rank, document_id in enumerate(rankings[sample_id], 1) if document_id in gold), None)
        if first is None:
            bins["not_found"] += 1
            continue
        found.append(first)
        label = "1" if first == 1 else "2_5" if first <= 5 else "6_10" if first <= 10 else "11_20" if first <= 20 else "21_50" if first <= 50 else "51_100" if first <= 100 else "beyond_100"
        bins[label] += 1
    labels = ("1", "2_5", "6_10", "11_20", "21_50", "51_100", "beyond_100", "not_found")
    return {
        "counts": {label: bins[label] for label in labels},
        "median_when_found": float(median(found)) if found else None,
    }


def metric_delta(alternative: dict, control: dict) -> dict:
    return {key: alternative[key] - control[key] for key in control if isinstance(control[key], float)}


def assert_dense_baseline(metrics: dict) -> None:
    observed = {key: metrics[key] for key in DENSE_BASELINE}
    delta = {key: observed[key] - DENSE_BASELINE[key] for key in DENSE_BASELINE}
    if max(abs(value) for value in delta.values()) > BASELINE_TOLERANCE:
        raise RuntimeError(f"fixed-window dense control mismatch: {delta}; stop before variants")


def save_result(result: dict) -> None:
    if result["split"]["evaluated_partition"] != "dev":
        raise RuntimeError("refusing to write a non-DEV result")
    RESULT_PATH.write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding="utf-8")
    print(json.dumps(result, ensure_ascii=False, indent=2))
    print("Saved aggregate-only DEV result:", RESULT_PATH)


In [ ]:
# Execute manually on Kaggle. Only overlap changes; chunk_size remains exactly 2000.
run_started = perf_counter()
if CHUNK_SIZE != 2_000 or OVERLAP_VARIANTS != {"overlap_0": 0, "overlap_200_control": 200, "overlap_500": 500}:
    raise RuntimeError("declared overlap axis changed")
dev, split_info = load_fixed_dev(LEGALIR_SOURCE_PATH)
if len(dev) != 1_036 or split_info["source_sha256"] != EXPECTED_SOURCE_SHA256:
    raise RuntimeError("fixed DEV identity mismatch")
documents = load_corpus(CORPUS_PATH)
dense = load_dense_model()
dense_metadata = dict(dense["metadata"])
query_encoding = encode_normalized_cls(dense, [sample["question"] for sample in dev.values()], QUERY_BATCH_SIZE)
sample_ids = list(dev)

results = {}
for label in ("overlap_200_control", "overlap_0", "overlap_500"):
    overlap = OVERLAP_VARIANTS[label]
    chunks, diagnostics = fixed_window_chunks(documents, CHUNK_SIZE, overlap)
    if label == "overlap_200_control" and len(chunks) != EXPECTED_FIXED_CHUNKS:
        raise RuntimeError("2000/200 control chunk count mismatch")
    encoding = encode_normalized_cls(dense, [chunk["text"] for chunk in chunks], CORPUS_BATCH_SIZE)
    retrieval = retrieve_dense_hits(query_encoding["embeddings"], encoding["embeddings"], chunks, sample_ids)
    candidates = candidates_from_hits(retrieval["hits"], "sum_top2", 200, support_limit=1, require_exact_depth=False)
    rankings = rankings_from_candidates(candidates)
    metrics = evaluate_rankings(dev, rankings, depths=EVALUATION_DEPTHS)
    if label == "overlap_200_control":
        assert_dense_baseline(metrics)
        for key, expected in DENSE_BASELINE.items():
            if abs(metrics[key] - expected) > BASELINE_TOLERANCE:
                raise RuntimeError(f"control {key} mismatch; STOP before interpreting overlap variants")
    results[label] = {
        "chunk_size": CHUNK_SIZE, "overlap": overlap, "step": CHUNK_SIZE - overlap,
        "chunk_count": len(chunks), "chunks_per_document": diagnostics["chunks_per_document"],
        "chunk_length_characters": diagnostics["chunk_length_characters"],
        "encoding_time_seconds": encoding["seconds"], "retrieval_time_seconds": retrieval["seconds"],
        "metrics": metrics,
    }
    del encoding, chunks, candidates, rankings
    gc.collect()
    torch.cuda.empty_cache()

control_metrics = results["overlap_200_control"]["metrics"]
result = {
    "split": split_info, "source_sha256": EXPECTED_SOURCE_SHA256, "query_count": len(dev),
    "research_axis": RESEARCH_AXIS, "control": CONTROL, "independent_variable": INDEPENDENT_VARIABLE,
    "fixed_components": FIXED_CONTROLS, "decision_rule": DECISION_RULE,
    "models": {
        "dense": {**dense_metadata, "name": DENSE_MODEL_NAME, "declared_revision": DENSE_DECLARED_REVISION,
                  "local_path": str(DENSE_MODEL_PATH), "local_files_only": True}
    },
    "chunk_candidate_support_invariants": {
        "source_preserving_exact_slices": True, "fixed_chunk_size": CHUNK_SIZE,
        "same_queries_and_corpus": True, "same_top_k_chunks": TOP_K_CHUNKS,
        "no_cross_encoder": True,
    },
    "control_metrics": control_metrics,
    "variant_metrics": {label: value["metrics"] for label, value in results.items() if label != "overlap_200_control"},
    "metric_deltas": {label: metric_delta(value["metrics"], control_metrics) for label, value in results.items() if label != "overlap_200_control"},
    "variants": results,
    "runtime": {"query_encoding_seconds": query_encoding["seconds"], "target_gpu": TARGET_GPU, "total_seconds": perf_counter() - run_started},
    "selection_interpretation": "Retrieval-only DEV diagnostic. No automatic winner or holdout decision.",
}
save_result(result)
